In [78]:
import sys
sys.path.append("..")

import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib.mlab   as mlab

from xkte_nonadaptive import kernel_dr_two_sample_test_agnostic
from dr_kte_adaptive import xMMD2_vsdr_fold_generic
from baselines import cadr_test, hadad_test, fit_krr
from sklearn.metrics import pairwise_distances

from scipy.spatial.distance import cdist
from scipy.special import expit
from scipy.stats import bernoulli
from numpy.polynomial.polynomial import polyval
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from scipy.stats import norm
import scipy.stats as stats
import statistics

from tqdm import tqdm

import seaborn as sns
import pandas as pd
import time
import os

In [93]:
data= pd.read_csv("https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/IHDP/csv/ihdp_npci_4.csv"
                  , header = None
                 )
col =  ["treatment", "y_factual", "y_cfactual", "mu0", "mu1" ,]
for i in range(1,26):
    col.append("x"+str(i))
data.columns = col
data = data.astype({"treatment":'bool'}, copy=False)
data

,treatment,y_factual,y_cfactual,mu0,mu1,x1,x2,x3,x4,x5,...,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25
0,True,12.698154,7.944765,7.962139,12.358607,-0.528603,-0.343455,1.128554,0.161703,-0.316603,...,1,1,1,1,0,0,0,0,0,0
1,False,9.806477,10.874676,8.405136,12.412752,-1.736945,-1.802002,0.383828,2.244320,-0.629189,...,1,1,1,1,0,0,0,0,0,0
2,False,8.268745,11.625162,7.571523,12.308303,-0.807451,-0.202946,-0.360898,-0.879606,0.808706,...,1,0,1,1,0,0,0,0,0,0
3,False,7.495897,11.810082,6.284592,12.122010,0.390083,0.596582,-1.850350,-0.879606,-0.004017,...,1,0,1,1,0,0,0,0,0,0
4,False,4.491960,12.322108,5.570813,12.001450,-1.045229,-0.602710,0.011465,0.161703,0.683672,...,1,1,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,False,9.960841,11.818981,8.918412,12.472027,-0.007654,-0.202946,-0.360898,0.161703,-0.316603,...,1,0,1,0,0,0,0,0,0,0
743,True,10.077599,9.115507,8.633288,12.439534,0.727295,-0.202946,-0.733261,-0.879606,0.808706,...,1,1,1,0,0,0,0,0,0,0
744,False,6.904068,13.067653,7.121436,12.247018,1.181234,0.196818,-1.477987,0.161703,0.746189,...,1,1,1,0,0,0,0,0,0,0
745,False,9.535070,11.797645,7.561980,12.307042,-0.288664,-0.202946,-1.477987,-0.879606,1.621430,...,1,1,1,0,0,0,0,0,0,0


In [84]:
df = pd.read_csv("ihdp.csv", index_col=0)
df

,iqsb.36,dose400,treat,bw,momage,nnhealth,birth.o,parity,moreprem,cigs,...,b.marryF,livwhoF,languageF,whenprenF,drugs.1,othstudy.1,momed4F,siteF,momraceF,workdur.imp.1
1,120.0,1,1,1559,33,94,2,2,1,0,...,1,1,1,1,2,2,4,1,w,1
2,90.0,0,1,1420,15,85,2,2,0,0,...,2,2,1,0,2,2,1,1,b,1
3,76.0,0,0,1000,33,89,4,5,0,20,...,3,2,1,1,2,2,1,1,b,1
4,43.0,0,0,1430,22,112,1,1,0,0,...,2,2,1,1,2,2,1,1,b,0
5,73.0,0,0,1984,20,99,1,1,0,10,...,2,2,1,2,2,2,3,1,b,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1086,105.0,0,1,2140,32,112,1,1,0,0,...,1,1,1,1,2,2,4,8,w,1
1087,73.0,0,0,2350,28,111,2,2,1,0,...,2,1,1,1,2,2,2,8,b,1
1088,127.0,0,0,1670,28,125,1,1,0,0,...,1,1,1,1,2,2,4,8,w,1
1089,98.0,0,0,1740,26,107,1,1,0,15,...,1,1,1,1,2,2,2,8,w,1


In [5]:
size_subset = 500
num_experiments = 20
iterations = 1000

b_list = ["I", "II", "III", "IV"]
method_list = []
np.random.seed(0)

# === Load IHDP ===
df = pd.read_csv("ihdp.csv", index_col=0)
covs_cont = [
    "bw",
    "momage",
    "nnhealth",
    "birth.o",
    "parity",
    "moreprem",
    "cigs",
    "alcohol",
    "ppvt.imp",
]
covs_cat = [
    "bwg",
    "female",
    "mlt.birt",
    "b.marry",
    "livwho",
    "language",
    "whenpren",
    "drugs",
    "othstudy",
]
features = covs_cont + covs_cat

df1 = df[features + ["iqsb.36", "treat"]].dropna()
scaler = StandardScaler()
df1[covs_cont] = scaler.fit_transform(df1[covs_cont])

X_all = df1[features].to_numpy()
T_all = df1["treat"].to_numpy()
Y_all = df1["iqsb.36"].to_numpy() / np.linalg.norm(df1["iqsb.36"].to_numpy())

In [23]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression

def treatment_effect_vector(ns, scenario, rng, beta_mix=1.0, beta_uniform=2.0):
    if scenario == 'I':
        return np.zeros(ns)
    if scenario == 'II':
        return np.full(ns, 1.0)
    if scenario == 'III':
        signs = rng.binomial(1, 0.5, size=ns) * 2 - 1
        return signs.astype(float) * beta_mix
    if scenario == 'IV':
        return rng.uniform(-beta_uniform, beta_uniform, size=ns)
    return np.zeros(ns)


def make_epsilon_greedy_on_ihdp_scenario(
    X_all, T_all, Y_all,
    ns=None,
    rng=None,
    eps0=0.5, eps_min=0.2, power=0.5,
    lam=1e-1,
    split="alternating",
    use_true_potentials=False,
    Y0_true=None, Y1_true=None,
    impute_method="ridge",
    add_resid_noise_scale=1.0,
    scenario='I',            # one of 'I','II','III','IV'
    beta_mix=1.0,
    beta_uniform=2.0,
):
    """
    IHDP epsilon-greedy run that injects a scenario-specific treatment effect delta[t]
    so the four scenarios I-IV from the synthetic experiments are reproduced.
    Returns same outputs as your synthetic collect_epsilon_greedy.
    """

    rng = rng or np.random.RandomState(0)

    N, d = X_all.shape
    if ns is None:
        ns = N
    assert ns <= N

    X = X_all[:ns].copy()
    T_obs = T_all[:ns].copy()
    Y_obs = Y_all[:ns].reshape(-1).copy()

    # --- Build base potential outcomes (impute if necessary) ---
    if use_true_potentials and (Y0_true is not None) and (Y1_true is not None):
        base_Y0 = np.asarray(Y0_true).reshape(-1)[:ns]
        base_Y1 = np.asarray(Y1_true).reshape(-1)[:ns]
    else:
        # Impute conditional means and simulate residual noise (same as before)
        def fit_model(X_tr, y_tr, method="ridge"):
            if method == "ridge":
                model = Ridge(alpha=1.0)
            else:
                model = LinearRegression()
            model.fit(X_tr, y_tr)
            return model

        mask0 = (T_obs == 0)
        mask1 = (T_obs == 1)

        if mask0.sum() >= 2:
            m0 = fit_model(X[mask0], Y_obs[mask0], method=impute_method)
            preds0 = m0.predict(X)
            resid0 = Y_obs[mask0] - m0.predict(X[mask0])
            sigma0 = np.std(resid0) if resid0.size > 0 else 1.0
        else:
            preds0 = np.full(ns, Y_obs.mean())
            sigma0 = np.std(Y_obs) * 0.5 if ns > 1 else 1.0

        if mask1.sum() >= 2:
            m1 = fit_model(X[mask1], Y_obs[mask1], method=impute_method)
            preds1 = m1.predict(X)
            resid1 = Y_obs[mask1] - m1.predict(X[mask1])
            sigma1 = np.std(resid1) if resid1.size > 0 else 1.0
        else:
            preds1 = np.full(ns, Y_obs.mean())
            sigma1 = np.std(Y_obs) * 0.5 if ns > 1 else 1.0

        base_Y0 = preds0 + rng.randn(ns) * (sigma0 * add_resid_noise_scale)
        base_Y1 = preds1 + rng.randn(ns) * (sigma1 * add_resid_noise_scale)

    # --- draw scenario-specific delta[t] and add to treated potential outcome ---
    delta = treatment_effect_vector(ns=ns, scenario=scenario, rng=rng,
                                    beta_mix=beta_mix, beta_uniform=beta_uniform)
    # Final potential outcomes used by the simulation:
    Y0_full = base_Y0
    Y1_full = base_Y1 + delta   # add scenario signal to treated arm

    # --- Online epsilon-greedy with per-arm ridge (same as your synthetic) ---
    X_aug = np.hstack([np.ones((ns, 1)), X])  # intercept + features
    S0 = np.diag([0.0] + [lam] * d)
    S1 = np.diag([0.0] + [lam] * d)
    b0 = np.zeros(d + 1)
    b1 = np.zeros(d + 1)

    def solve_theta(S, b):
        try:
            return np.linalg.solve(S, b)
        except np.linalg.LinAlgError:
            return np.linalg.lstsq(S, b, rcond=None)[0]

    def eps_at(t):
        return max(eps_min, eps0 / ((t + 1) ** power))

    T_sim = np.zeros(ns, dtype=int)
    w_decision = np.zeros(ns, dtype=float)   # decision-time probability for action=1
    Y_sim = np.zeros(ns, dtype=float)
    theta0_snap = np.zeros((ns, d + 1))
    theta1_snap = np.zeros((ns, d + 1))

    for t in range(ns):
        th0 = solve_theta(S0, b0)
        th1 = solve_theta(S1, b1)
        theta0_snap[t] = th0
        theta1_snap[t] = th1

        eps_t = eps_at(t)
        z_t = X_aug[t]
        q0 = z_t @ th0
        q1 = z_t @ th1
        if q1 > q0:
            pi1 = 1.0 - 0.5 * eps_t
        elif q1 < q0:
            pi1 = 0.5 * eps_t
        else:
            pi1 = 0.5

        a = 1 if rng.rand() < pi1 else 0
        T_sim[t] = a
        w_decision[t] = pi1

        y_t = Y1_full[t] if a == 1 else Y0_full[t]
        Y_sim[t] = y_t

        if a == 0:
            S0 += np.outer(z_t, z_t)
            b0 += z_t * y_t
        else:
            S1 += np.outer(z_t, z_t)
            b1 += z_t * y_t

    # --- fold splits and fold×fold policy matrices (chrono × chrono) ---
    if split == "alternating":
        idx0 = np.arange(0, ns, 2)
        idx1 = np.arange(1, ns, 2)
    elif split == "chronological":
        cut = ns // 2
        idx0 = np.arange(0, cut)
        idx1 = np.arange(cut, ns)
    else:
        raise ValueError("split must be 'alternating' or 'chronological'")

    N0, N1 = idx0.size, idx1.size
    Z0 = X_aug[idx0]
    Z1 = X_aug[idx1]

    Pi_fold0_on_0 = np.empty((N0, N0))
    for r, t in enumerate(idx0):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0_all = Z0 @ th0
        q1_all = Z0 @ th1
        Pi_fold0_on_0[r] = np.where(q1_all > q0_all,
                                    1.0 - 0.5 * eps_t,
                                    np.where(q1_all < q0_all, 0.5 * eps_t, 0.5))

    Pi_fold1_on_1 = np.empty((N1, N1))
    for r, t in enumerate(idx1):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0_all = Z1 @ th0
        q1_all = Z1 @ th1
        Pi_fold1_on_1[r] = np.where(q1_all > q0_all,
                                    1.0 - 0.5 * eps_t,
                                    np.where(q1_all < q0_all, 0.5 * eps_t, 0.5))

    P_all = np.empty((ns, ns), dtype=np.float32)
    for t in range(ns):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0_all = X_aug @ th0
        q1_all = X_aug @ th1
        P_all[t] = np.where(q1_all > q0_all,
                            1.0 - 0.5 * eps_t,
                            np.where(q1_all < q0_all, 0.5 * eps_t, 0.5))

    return X, T_sim, Y_sim.reshape(-1, 1), w_decision, Pi_fold0_on_0, Pi_fold1_on_1, idx0, idx1, P_all


In [72]:
rng = np.random.RandomState(42)
rng = np.random.RandomState(np.random.randint(500000))
ns = 800
X_sub, T_sim, Y_sim, w_decision, Pi_0_on_0, Pi_1_on_1, idx0, idx1, P_all = \
    make_epsilon_greedy_on_ihdp_scenario(
        X_all, T_all, Y_all,
        ns=800,
        rng=rng,
        eps0=0.5, eps_min=0.2, power=0.5,
        lam=1e-1,
        split="alternating",
        use_true_potentials=False,   # True if you have Y0_true,Y1_true
        impute_method="ridge",
        add_resid_noise_scale=1.0,
        scenario='I',              # choose 'I','II','III','IV'
        beta_mix=1.0,
        beta_uniform=2.0
    )

# Now pass these to your VS-DR test as you do for synthetic:
sigma2 = np.median(pairwise_distances(Y_sim, Y_sim)) ** 2
gamma_k = 1.0 / sigma2 if sigma2 > 0 else None

stat = xMMD2_vsdr_fold_generic(
    Y=Y_sim, w=w_decision, X=X_sub, A=T_sim,
    kernel_function='rbf',
    Pi_0_on_0=Pi_0_on_0, Pi_1_on_1=Pi_1_on_1,
    idx0=idx0, idx1=idx1, gamma=gamma_k
)


In [74]:
stat

4.817797821599708

In [77]:
T_sim, T_all;

(array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
        1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
        1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
        0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1,
        0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1,
        1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 